# 💊 Bi-modal Cross-Attention Drug Sensitivity Demo
### *Predicting Cancer Cell Line $\ln(\mathrm{IC}_{50})$ & Mechanistic Pathway Attention*

This interactive tutorial demonstrates how the **Cross-Attention Drug Synergy** Transformer fuses:
1. **Small-Molecule Chemical Structures (SMILES)** via a Multi-Head Self-Attention Transformer.
2. **Tumor Transcriptomics (RNA-seq)** structured across 8 hallmark oncogenic pathways.
3. **Bi-directional Cross-Attention** allowing chemical warheads to query tumor oncogenes.

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import CURATED_DRUGS, REPRESENTATIVE_CELL_LINES, PATHWAY_NAMES, MODEL_CONFIG
from src.smiles_tokenizer import SMILESTokenizer
from src.cross_attention_transformer import CrossAttentionDrugSensitivityModel
from src.data_loader import generate_synthetic_gdsc_data
from src.visualizer import CrossAttentionVisualizer

print("PyTorch Version:", torch.__version__)
print(f"Available Curated Drugs: {len(CURATED_DRUGS)}")
print(f"Hallmark Cancer Pathways: {len(PATHWAY_NAMES)}")

## 1. Inspect Curated Oncology Drugs & Hallmark Pathways

In [ ]:
for drug, info in list(CURATED_DRUGS.items())[:5]:
    print(f"• {drug:<12} | Target: {info['target_pathway']:<20} | Class: {info['drug_class']}")
    print(f"  SMILES: {info['smiles'][:60]}...")

## 2. Load the Pre-Trained Model Checkpoint

In [ ]:
tokenizer = SMILESTokenizer(max_length=MODEL_CONFIG['max_smiles_len'])
model = CrossAttentionDrugSensitivityModel(MODEL_CONFIG)

ckpt_path = '../checkpoints/best_cross_attention_model.pt'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    print(f"Loaded trained weights from: {ckpt_path}")
else:
    print("Using randomly initialized model (run train_and_evaluate.py to train!)")
model.eval()

## 3. Predict Drug Sensitivity for Osimertinib in EGFR-Mutant PC-9

In [ ]:
drug_name = 'Osimertinib'
cell_line = 'PC-9'
smiles = CURATED_DRUGS[drug_name]['smiles']

# Create sample data record
records = generate_synthetic_gdsc_data(num_samples=100, random_seed=42)
pc9_record = [r for r in records if r['cell_line'] == cell_line][0]

token_ids, mask = tokenizer.encode(smiles)
gene_expr = torch.tensor(pc9_record['gene_expression'], dtype=torch.float32).unsqueeze(0)

# Extract attention maps and prediction
maps = model.extract_attention_maps(token_ids.unsqueeze(0), mask.unsqueeze(0), gene_expr)
pred_ln_ic50 = maps['predicted_ic50'].item()
pred_ic50_um = np.exp(pred_ln_ic50)

print(f"Target Drug:             {drug_name}")
print(f"Cancer Cell Line:        {cell_line} ({pc9_record['cancer_type']})")
print(f"Predicted ln(IC50):      {pred_ln_ic50:.3f} ln(uM)")
print(f"Predicted IC50 in uM:    {pred_ic50_um:.4f} uM ({pred_ic50_um*1000:.1f} nM)")

## 4. Visualize the Bi-modal Cross-Attention Map

In [ ]:
# Average across 4 attention heads
d2p_attn = maps['drug_to_pathway_attn'][0].mean(dim=0).cpu().numpy()
valid_len = mask.sum().item()
tokens = [tokenizer.CLS_TOKEN] + tokenizer.tokenize(smiles)

viz = CrossAttentionVisualizer(output_dir='../figures')
fig_path = viz.plot_cross_attention_heatmap(
    attn_matrix=d2p_attn[:valid_len, :],
    smiles_tokens=tokens[:valid_len],
    drug_name=drug_name,
    cell_line=cell_line,
    filename='demo_cross_attention_heatmap.png'
)

from IPython.display import Image
Image(fig_path)